## PubChem property enrichment + Ro5/BBB preparation

In [1]:
import pandas as pd
import requests
import time

df = pd.read_csv("jupyter_output/kg_compounds_with_pubchem.csv")

print(df.shape)
df.head()

(216, 12)


,gene,compound,compound_code,cui,relationship,source,pubchem_cid,canonical_smiles,molecular_formula,molecular_weight,iupac_name,pubchem_status
0,NR1H2 gene,(R)-5-{4-[2-(Methyl-pyridin-2-yl-amino)-ethoxy...,10021239,C0289313,negatively_correlated_with_chemical_or_drug,CMAP,NaN,NaN,NaN,NaN,NaN,CID not found
1,NR1H2 gene,(R)-5-{4-[2-(Methyl-pyridin-2-yl-amino)-ethoxy...,10021239,C0289313,inverse_negatively_correlated_with_chemical_or...,CMAP,NaN,NaN,NaN,NaN,NaN,CID not found
2,NR1H2 gene,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",4302962,PUBCHEM:4302962 CUI,negatively_regulates,LINCS,4302962.0,NaN,C21H19Cl2IN4O2,557.20,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",found
3,NR1H2 gene,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",4302962,PUBCHEM:4302962 CUI,negatively_regulated_by,LINCS,4302962.0,NaN,C21H19Cl2IN4O2,557.20,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",found
4,NR1H2 gene,1-phenylbiguanide,4780,PUBCHEM:4780 CUI,positively_regulates,LINCS,4780.0,NaN,C8H11N5,177.21,1-(diaminomethylidene)-2-phenylguanidine,found


In [2]:
###pull PubChem properties by CID
props = [
    "MolecularWeight",
    "XLogP",
    "TPSA",
    "HBondDonorCount",
    "HBondAcceptorCount",
    "RotatableBondCount",
    "SMILES"
]

def get_pubchem_props_by_cid(cid):
    if pd.isna(cid):
        return {}

    cid = int(cid)
    prop_string = ",".join(props)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/{prop_string}/JSON"

    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return {"pubchem_cid": cid, "property_status": "not_found"}

        data = r.json()["PropertyTable"]["Properties"][0]
        data["pubchem_cid"] = cid
        data["property_status"] = "found"
        return data

    except Exception as e:
        return {"pubchem_cid": cid, "property_status": f"error: {e}"}

In [3]:
#run for unique CID
unique_cids = (
    df["pubchem_cid"]
    .dropna()
    .astype(int)
    .drop_duplicates()
    .tolist()
)

print("Unique CIDs:", len(unique_cids))

Unique CIDs: 75


In [4]:

results = []

for i, cid in enumerate(unique_cids, start=1):
    print(f"{i}/{len(unique_cids)} CID {cid}")
    results.append(get_pubchem_props_by_cid(cid))
    time.sleep(0.2)

props_df = pd.DataFrame(results)

props_df.head()

1/75 CID 4302962
2/75 CID 4780
3/75 CID 5073
4/75 CID 4993
5/75 CID 2708
6/75 CID 2895
7/75 CID 451668
8/75 CID 3162
9/75 CID 83933
10/75 CID 5756
11/75 CID 5870
12/75 CID 57363
13/75 CID 3559
14/75 CID 3830
15/75 CID 149096
16/75 CID 28864
17/75 CID 4184
18/75 CID 5360515
19/75 CID 156391
20/75 CID 4908
21/75 CID 1978
22/75 CID 2160
23/75 CID 9330
24/75 CID 448537
25/75 CID 12560
26/75 CID 33478
27/75 CID 5755
28/75 CID 5865
29/75 CID 5833
30/75 CID 444795
31/75 CID 3158
32/75 CID 24066
33/75 CID 4913
34/75 CID 1989
35/75 CID 135398513
36/75 CID 5280965
37/75 CID 2337
38/75 CID 3025
39/75 CID 5280961
40/75 CID 3478
41/75 CID 6032
42/75 CID 896
43/75 CID 15387
44/75 CID 441130
45/75 CID 126941
46/75 CID 4543
47/75 CID 4829
48/75 CID 43672
49/75 CID 5770
50/75 CID 5320
51/75 CID 5323
52/75 CID 5325
53/75 CID 5533
54/75 CID 135398737
55/75 CID 3446
56/75 CID 3658
57/75 CID 4421
58/75 CID 8955
59/75 CID 3290
60/75 CID 5253
61/75 CID 6769
62/75 CID 5403
63/75 CID 5273569
64/75 CID 2122
65/

,CID,MolecularWeight,SMILES,XLogP,TPSA,HBondDonorCount,HBondAcceptorCount,RotatableBondCount,pubchem_cid,property_status
0,4302962,557.2,CC1=C(N(N=C1C(=O)NN2CCOCC2)C3=C(C=C(C=C3)Cl)Cl...,5.3,59.4,1,4,4,4302962,found
1,4780,177.21,C1=CC=C(C=C1)N=C(N)N=C(N)N,0.0,103.0,3,1,2,4780,found
2,5073,410.5,CC1=C(C(=O)N2CCCCC2=N1)CCN3CCC(CC3)C4=NOC5=C4C...,2.7,61.9,0,6,4,5073,found
3,4993,248.71,CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl,2.7,77.8,2,4,2,4993,found
4,2708,304.2,C1=CC(=CC=C1CCCC(=O)O)N(CCCl)CCCl,1.7,40.5,1,3,9,2708,found


In [5]:
props_df.to_csv("jupyter_output/pubchem_property_enrichment.csv", index=False)
print("Saved: pubchem_property_enrichment.csv")

Saved: pubchem_property_enrichment.csv


#### Merge properties back to KG compounds

In [6]:
df["pubchem_cid"] = pd.to_numeric(df["pubchem_cid"], errors="coerce")
props_df["pubchem_cid"] = pd.to_numeric(props_df["pubchem_cid"], errors="coerce")

enriched = df.merge(props_df, on="pubchem_cid", how="left")

print(enriched.shape)
enriched.head()

(216, 21)


,gene,compound,compound_code,cui,relationship,source,pubchem_cid,canonical_smiles,molecular_formula,molecular_weight,...,pubchem_status,CID,MolecularWeight,SMILES,XLogP,TPSA,HBondDonorCount,HBondAcceptorCount,RotatableBondCount,property_status
0,NR1H2 gene,(R)-5-{4-[2-(Methyl-pyridin-2-yl-amino)-ethoxy...,10021239,C0289313,negatively_correlated_with_chemical_or_drug,CMAP,NaN,NaN,NaN,NaN,...,CID not found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NR1H2 gene,(R)-5-{4-[2-(Methyl-pyridin-2-yl-amino)-ethoxy...,10021239,C0289313,inverse_negatively_correlated_with_chemical_or...,CMAP,NaN,NaN,NaN,NaN,...,CID not found,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NR1H2 gene,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",4302962,PUBCHEM:4302962 CUI,negatively_regulates,LINCS,4302962.0,NaN,C21H19Cl2IN4O2,557.20,...,found,4302962.0,557.2,CC1=C(N(N=C1C(=O)NN2CCOCC2)C3=C(C=C(C=C3)Cl)Cl...,5.3,59.4,1.0,4.0,4.0,found
3,NR1H2 gene,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",4302962,PUBCHEM:4302962 CUI,negatively_regulated_by,LINCS,4302962.0,NaN,C21H19Cl2IN4O2,557.20,...,found,4302962.0,557.2,CC1=C(N(N=C1C(=O)NN2CCOCC2)C3=C(C=C(C=C3)Cl)Cl...,5.3,59.4,1.0,4.0,4.0,found
4,NR1H2 gene,1-phenylbiguanide,4780,PUBCHEM:4780 CUI,positively_regulates,LINCS,4780.0,NaN,C8H11N5,177.21,...,found,4780.0,177.21,C1=CC=C(C=C1)N=C(N)N=C(N)N,0.0,103.0,3.0,1.0,2.0,found


In [7]:
enriched.to_csv("jupyter_output/kg_compounds_pubchem_enriched.csv", index=False)
print("Saved: kg_compounds_pubchem_enriched.csv")

Saved: kg_compounds_pubchem_enriched.csv


#### --For myself: Make Ro5 file for the web app--

In [8]:
ro5_file = (
    enriched[["pubchem_cid", "SMILES"]]
    .drop_duplicates()
    .copy()
)

ro5_file = ro5_file[["pubchem_cid", "SMILES"]]
ro5_file = ro5_file.dropna(subset=["SMILES"])

ro5_file.to_csv("jupyter_output/compound_smiles_for_ro5.csv", index=False)

print(ro5_file.shape)
ro5_file.head()

(75, 2)


,pubchem_cid,SMILES
2,4302962.0,CC1=C(N(N=C1C(=O)NN2CCOCC2)C3=C(C=C(C=C3)Cl)Cl...
4,4780.0,C1=CC=C(C=C1)N=C(N)N=C(N)N
6,5073.0,CC1=C(C(=O)N2CCCCC2=N1)CCN3CCC(CC3)C4=NOC5=C4C...
8,4993.0,CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl
12,2708.0,C1=CC(=CC=C1CCCC(=O)O)N(CCCl)CCCl


## Rule-of-5 + rough BBB flags

In [9]:
screen = enriched.copy()

for col in ["MolecularWeight", "XLogP", "TPSA", "HBondDonorCount", "HBondAcceptorCount"]:
    screen[col] = pd.to_numeric(screen[col], errors="coerce")

screen["ro5_mw_ok"] = screen["MolecularWeight"] < 500
screen["ro5_xlogp_ok"] = screen["XLogP"] <= 5
screen["ro5_hbd_ok"] = screen["HBondDonorCount"] <= 5
screen["ro5_hba_ok"] = screen["HBondAcceptorCount"] <= 10

screen["ro5_pass_count"] = screen[
    ["ro5_mw_ok", "ro5_xlogp_ok", "ro5_hbd_ok", "ro5_hba_ok"]
].sum(axis=1)

screen["rough_ro5_pass"] = screen["ro5_pass_count"] >= 3

# Rough BBB-friendly filter
screen["bbb_mw_ok"] = screen["MolecularWeight"] < 500
screen["bbb_xlogp_ok"] = screen["XLogP"].between(1, 5, inclusive="both")
screen["bbb_tpsa_ok"] = screen["TPSA"] < 90
screen["bbb_hbd_ok"] = screen["HBondDonorCount"] <= 3

screen["bbb_pass_count"] = screen[
    ["bbb_mw_ok", "bbb_xlogp_ok", "bbb_tpsa_ok", "bbb_hbd_ok"]
].sum(axis=1)

screen["rough_bbb_candidate"] = screen["bbb_pass_count"] >= 3

screen.head()

,gene,compound,compound_code,cui,relationship,source,pubchem_cid,canonical_smiles,molecular_formula,molecular_weight,...,ro5_hbd_ok,ro5_hba_ok,ro5_pass_count,rough_ro5_pass,bbb_mw_ok,bbb_xlogp_ok,bbb_tpsa_ok,bbb_hbd_ok,bbb_pass_count,rough_bbb_candidate
0,NR1H2 gene,(R)-5-{4-[2-(Methyl-pyridin-2-yl-amino)-ethoxy...,10021239,C0289313,negatively_correlated_with_chemical_or_drug,CMAP,NaN,NaN,NaN,NaN,...,False,False,0,False,False,False,False,False,0,False
1,NR1H2 gene,(R)-5-{4-[2-(Methyl-pyridin-2-yl-amino)-ethoxy...,10021239,C0289313,inverse_negatively_correlated_with_chemical_or...,CMAP,NaN,NaN,NaN,NaN,...,False,False,0,False,False,False,False,False,0,False
2,NR1H2 gene,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",4302962,PUBCHEM:4302962 CUI,negatively_regulates,LINCS,4302962.0,NaN,C21H19Cl2IN4O2,557.20,...,True,True,2,False,False,False,True,True,2,False
3,NR1H2 gene,"1-(2,4-dichlorophenyl)-5-(4-iodophenyl)-4-meth...",4302962,PUBCHEM:4302962 CUI,negatively_regulated_by,LINCS,4302962.0,NaN,C21H19Cl2IN4O2,557.20,...,True,True,2,False,False,False,True,True,2,False
4,NR1H2 gene,1-phenylbiguanide,4780,PUBCHEM:4780 CUI,positively_regulates,LINCS,4780.0,NaN,C8H11N5,177.21,...,True,True,4,True,True,False,False,True,2,False


In [10]:
#Save screening table
screen.to_csv("jupyter_output/kg_compounds_ro5_bbb_screened.csv", index=False)
print("Saved: kg_compounds_ro5_bbb_screened.csv")

Saved: kg_compounds_ro5_bbb_screened.csv


### Quick summary target

In [11]:
bbb_summary = (
    screen.groupby("gene")
    .agg(
        total_rows=("compound", "count"),
        unique_compounds=("compound", "nunique"),
        ro5_pass_compounds=("rough_ro5_pass", "sum"),
        rough_bbb_candidate_rows=("rough_bbb_candidate", "sum")
    )
    .reset_index()
)

bbb_summary


,gene,total_rows,unique_compounds,ro5_pass_compounds,rough_bbb_candidate_rows
0,NR1H2 gene,86,42,56,52
1,NR1H3 gene,86,43,70,52
2,TYROBP gene,44,22,26,20


In [12]:
bbb_summary.to_csv("jupyter_output/bbb_ro5_summary_by_target.csv", index=False)
print("Saved: bbb_ro5_summary_by_target.csv")

Saved: bbb_ro5_summary_by_target.csv


### Top Candidate list for next step.

In [13]:
top_candidates = (
    screen[
        ["gene", "compound", "pubchem_cid", "MolecularWeight", "XLogP", "TPSA",
         "HBondDonorCount", "HBondAcceptorCount", "SMILES",
         "rough_ro5_pass", "rough_bbb_candidate"]
    ]
    .drop_duplicates()
    .sort_values(["rough_bbb_candidate", "rough_ro5_pass", "gene"], ascending=[False, False, True])
)

top_candidates.to_csv("jupyter_output/top_ro5_bbb_candidates.csv", index=False)

top_candidates.head(30)

,gene,compound,pubchem_cid,MolecularWeight,XLogP,TPSA,HBondDonorCount,HBondAcceptorCount,SMILES,rough_ro5_pass,rough_bbb_candidate
6,NR1H2 gene,3-{2-[4-(6-Fluoro-benzo[d]isoxazol-3-yl)-piper...,5073.0,410.50,2.7,61.9,0.0,6.0,CC1=C(C(=O)N2CCCCC2=N1)CCN3CCC(CC3)C4=NOC5=C4C...,True,True
8,NR1H2 gene,"5-(4-Chloro-phenyl)-6-ethyl-pyrimidine-2,4-dia...",4993.0,248.71,2.7,77.8,2.0,4.0,CCC1=C(C(=NC(=N1)N)N)C2=CC=C(C=C2)Cl,True,True
12,NR1H2 gene,Chlorambucil,2708.0,304.20,1.7,40.5,1.0,3.0,C1=CC(=CC=C1CCCC(=O)O)N(CCCl)CCCl,True,True
14,NR1H2 gene,Cyclobenzaprine,2895.0,275.40,5.2,3.2,0.0,1.0,CN(C)CCC=C1C2=CC=CC=C2C=CC3=CC=CC=C31,True,True
18,NR1H2 gene,Doxylamine,3162.0,270.37,2.5,25.4,0.0,3.0,CC(C1=CC=CC=C1)(C2=CC=CC=N2)OCCN(C)C,True,True
22,NR1H2 gene,Estriol,5756.0,288.40,2.5,60.7,3.0,3.0,C[C@]12CC[C@H]3[C@H]([C@@H]1C[C@H]([C@@H]2O)O)...,True,True
24,NR1H2 gene,Estrone,5870.0,270.40,3.1,37.3,1.0,2.0,C[C@]12CC[C@H]3[C@H]([C@@H]1CCC2=O)CCC4=C3C=CC...,True,True
26,NR1H2 gene,Finasteride,57363.0,372.50,3.0,58.2,2.0,2.0,C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2C(=O)NC(C...,True,True
28,NR1H2 gene,Haloperidol,3559.0,375.90,3.2,40.5,1.0,4.0,C1CN(CCC1(C2=CC=C(C=C2)Cl)O)CCCC(=O)C3=CC=C(C=...,True,True
30,NR1H2 gene,Kinetin,3830.0,215.21,1.0,79.6,2.0,5.0,C1=COC(=C1)CNC2=NC=NC3=C2NC=N3,True,True


In [14]:
print(enriched["compound"].nunique()) # total unique compounds
print(enriched["SMILES"].isna().sum()) # how many lost to missing SMILES
print(enriched.dropna(subset=["SMILES"])["compound"].nunique())  # should ≈ 69

99
52
75
